# AI Hub 경구약제 단일 이미지 데이터셋 EDA

AI Hub 경구약제 이미지 데이터셋(5,000종) 중 단일 알약 데이터(TL_1/TS_1)를 다운로드하고 탐색합니다.  
**조합 데이터(TL_2/TS_2)는 Kaggle 대회 데이터 원본이므로 절대 사용하지 않습니다.**

## 1. AI Hub 데이터 다운로드

Colab Secrets에 저장된 `AIHUB_ID`(닉네임)와 `AIHUB_KEY`(토큰)로 인증합니다.  
**Colab Secrets 등록 방법**: 좌측 자물쇠 아이콘 → `AIHUB_ID`, `AIHUB_KEY` 추가

In [ ]:
# aihubshell 설치 (최초 1회)
!pip install -q aihubshell

In [ ]:
import os
from pathlib import Path

from google.colab import userdata

AIHUB_ID  = userdata.get('AIHUB_ID')
AIHUB_KEY = userdata.get('AIHUB_KEY')

AIHUB_ROOT = Path('/content/aihub_pills')
AIHUB_ROOT.mkdir(parents=True, exist_ok=True)

print('AIHUB_ID 확인:', AIHUB_ID[:3] + '***' if AIHUB_ID else '없음')
print('저장 경로:', AIHUB_ROOT)

In [ ]:
# 데이터셋 576번 파일 목록 확인 — 실제 filekey 이름 확인용
!aihubshell -mode l -datasetkey 576 -id {AIHUB_ID} -pass {AIHUB_KEY}

In [ ]:
# TL_1(라벨/단일), TS_1(이미지/단일) 다운로드
# 위 목록 확인 후 실제 filekey 값으로 수정
os.chdir(AIHUB_ROOT)

!aihubshell -mode d -datasetkey 576 -filekey TL_1 -id {AIHUB_ID} -pass {AIHUB_KEY}
!aihubshell -mode d -datasetkey 576 -filekey TS_1 -id {AIHUB_ID} -pass {AIHUB_KEY}

In [ ]:
import subprocess

for pattern in ['TL_1*.zip', 'TS_1*.zip']:
    parts = sorted(AIHUB_ROOT.glob(pattern))
    if not parts:
        print(f'{pattern} 파일 없음, 스킵')
        continue
    if len(parts) == 1:
        subprocess.run(['unzip', '-o', str(parts[0]), '-d', str(AIHUB_ROOT)], check=True)
    else:
        merged = AIHUB_ROOT / pattern.replace('*.zip', '_merged.zip')
        with merged.open('wb') as f:
            for p in parts:
                f.write(p.read_bytes())
        subprocess.run(['unzip', '-o', str(merged), '-d', str(AIHUB_ROOT)], check=True)
    print(f'{pattern} 압축 해제 완료')

print('\n디렉토리 구조:')
!find {AIHUB_ROOT} -maxdepth 3 -type d | head -30

## 2. 데이터 기본 정보 확인

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

image_paths = sorted(AIHUB_ROOT.rglob('*.png'))
json_paths  = sorted(AIHUB_ROOT.rglob('*.json'))

print(f'총 이미지 수: {len(image_paths):,}')
print(f'총 JSON 수  : {len(json_paths):,}')

# 이미지 크기 분포
size_rows = []
for p in image_paths[:500]:  # 샘플링 (전체가 너무 크면)
    with Image.open(p) as img:
        size_rows.append({'width': img.width, 'height': img.height})

size_df = pd.DataFrame(size_rows).groupby(['width', 'height']).size().reset_index(name='count')
print('\n이미지 크기 분포 (샘플 500장 기준):')
display(size_df)

# 샘플 JSON 구조
if json_paths:
    with json_paths[0].open(encoding='utf-8') as f:
        sample = json.load(f)
    print('\nJSON 최상위 키:', list(sample.keys()))
    if sample.get('categories'):
        print('카테고리 샘플:', sample['categories'][0])
    if sample.get('annotations'):
        print('어노테이션 샘플:', sample['annotations'][0])

In [ ]:
# 전체 JSON에서 카테고리 + 어노테이션 수집
category_by_id = {}
form_by_id     = {}
bbox_rows      = []

for json_path in json_paths:
    with json_path.open(encoding='utf-8') as f:
        data = json.load(f)

    image_id_to_info = {img['id']: img for img in data.get('images', [])}

    for cat in data.get('categories', []):
        category_by_id[cat['id']] = cat.get('name', '')
        form_by_id[cat['id']] = (
            cat.get('pill_form') or cat.get('form') or cat.get('drug_form') or '미분류'
        )

    for ann in data.get('annotations', []):
        img_info = image_id_to_info.get(ann['image_id'], {})
        img_w = img_info.get('width', 0)
        img_h = img_info.get('height', 0)
        x, y, bw, bh = ann.get('bbox', [0, 0, 0, 0])

        bbox_rows.append({
            'annotation_id'  : ann.get('id'),
            'image_id'       : ann.get('image_id'),
            'image_file_name': img_info.get('file_name', ''),
            'class_id'       : ann.get('category_id'),
            'image_width'    : img_w,
            'image_height'   : img_h,
            'bbox_x'         : x,
            'bbox_y'         : y,
            'bbox_width'     : bw,
            'bbox_height'    : bh,
            'bbox_area'      : bw * bh,
            'aspect_ratio'   : bw / bh if bh else None,
            'center_x'       : x + bw / 2,
            'center_y'       : y + bh / 2,
            'bbox_width_ratio' : bw / img_w if img_w else None,
            'bbox_height_ratio': bh / img_h if img_h else None,
            'bbox_area_ratio'  : (bw * bh) / (img_w * img_h) if img_w and img_h else None,
            'center_x_ratio'   : (x + bw / 2) / img_w if img_w else None,
            'center_y_ratio'   : (y + bh / 2) / img_h if img_h else None,
        })

class_df = pd.DataFrame([
    {'class_id': cid, 'class_name': name, 'pill_form': form_by_id.get(cid, '미분류')}
    for cid, name in sorted(category_by_id.items())
])
bbox_df = pd.DataFrame(bbox_rows).merge(class_df, on='class_id', how='left')

print(f'총 클래스 수    : {len(class_df):,}')
print(f'총 어노테이션 수: {len(bbox_df):,}')
display(class_df.head())

## 3. 클래스별 데이터 분포 확인

클래스당 이미지 수 분포를 KDE plot / Deviation / Box plot 세 가지로 확인합니다.

In [ ]:
class_dist = (
    bbox_df.drop_duplicates(['class_id', 'image_id'])
    .groupby('class_id').size().reset_index(name='image_count')
    .merge(class_df, on='class_id', how='left')
    .sort_values('image_count', ascending=False)
)

image_count_values = class_dist['image_count']
ideal_count        = image_count_values.sum() / len(class_dist)

dist_summary = pd.DataFrame([
    {'metric': 'mean',                  'value': image_count_values.mean()},
    {'metric': 'median',                'value': image_count_values.median()},
    {'metric': 'std',                   'value': image_count_values.std()},
    {'metric': 'skew',                  'value': image_count_values.skew()},
    {'metric': 'ideal_uniform_count',   'value': ideal_count},
])
display(dist_summary)
display(class_dist.head(20))

In [ ]:
# KDE plot
kde_x = np.linspace(image_count_values.min(), image_count_values.max(), 500)
bw    = 1.06 * image_count_values.std() * len(image_count_values) ** (-1 / 5)
kde_y = sum(np.exp(-0.5 * ((kde_x - v) / bw) ** 2) for v in image_count_values)
kde_y /= len(image_count_values) * bw * np.sqrt(2 * np.pi)

plt.figure(figsize=(10, 5))
plt.hist(image_count_values, bins=40, density=True, alpha=0.6, label='Class count histogram')
plt.plot(kde_x, kde_y, color='black', linewidth=2, label='KDE')
plt.axvline(image_count_values.mean(),   color='red',   linestyle='--', label=f'Mean: {image_count_values.mean():.1f}')
plt.axvline(image_count_values.median(), color='blue',  linestyle='--', label=f'Median: {image_count_values.median():.1f}')
plt.axvline(ideal_count,                 color='green', linestyle='-',  label=f'Ideal uniform: {ideal_count:.1f}')
plt.title('Class Image Count Distribution with KDE (AI Hub 5,000종)')
plt.xlabel('Image count per class')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Deviation from ideal — 상위/하위 30개만 표시 (5000종 전체 표시 불가)
deviation_df = class_dist.copy()
deviation_df['deviation'] = deviation_df['image_count'] - ideal_count

top30    = deviation_df.nlargest(30, 'deviation')
bottom30 = deviation_df.nsmallest(30, 'deviation')
plot_df  = pd.concat([bottom30, top30]).drop_duplicates().sort_values('deviation')

colors = plot_df['deviation'].apply(lambda v: 'tab:red' if v > 0 else 'tab:blue')

plt.figure(figsize=(10, 14))
plt.barh(plot_df['class_name'], plot_df['deviation'], color=colors)
plt.axvline(0, color='black', linewidth=1)
plt.title(f'Deviation from Ideal Uniform Count ({ideal_count:.1f}) — 상/하위 30클래스')
plt.xlabel('Deviation from ideal')
plt.ylabel('Class name')
plt.tight_layout()
plt.show()

In [ ]:
# Box plot
plt.figure(figsize=(5, 7))
plt.boxplot(image_count_values, vert=True)
plt.axhline(ideal_count,                 color='green', linestyle='-',  label=f'Ideal uniform: {ideal_count:.1f}')
plt.axhline(image_count_values.median(), color='blue',  linestyle='--', label=f'Median: {image_count_values.median():.1f}')
plt.title('Class Image Count Box Plot')
plt.ylabel('Image count per class')
plt.xticks([1], ['Classes'])
plt.legend()
plt.tight_layout()
plt.show()

## 4. 약제 형태별 분포 (정제 / 경질캡슐 / 연질캡슐)

Kaggle 데이터에 없던 형태 정보를 AI Hub 데이터에서 확인합니다.

In [ ]:
form_dist = (
    class_df.groupby('pill_form').size().reset_index(name='class_count')
    .sort_values('class_count', ascending=False)
)
form_dist['ratio'] = form_dist['class_count'] / form_dist['class_count'].sum()
display(form_dist)

plt.figure(figsize=(7, 4))
plt.bar(form_dist['pill_form'], form_dist['class_count'], color='steelblue', alpha=0.8)
plt.title('약제 형태별 클래스 수')
plt.xlabel('약제 형태')
plt.ylabel('클래스 수')
plt.tight_layout()
plt.show()

## 5. 클래스 간 시각적 유사성

형태별 샘플 이미지를 확인해 Kaggle 데이터와 도메인 갭이 있는지 살펴봅니다.

In [ ]:
image_path_by_name = {p.name: p for p in image_paths}

# JSON에서 image_id → file_name 매핑 구축
image_file_by_id = {}
for jp in json_paths:
    with jp.open(encoding='utf-8') as f:
        d = json.load(f)
    for img_meta in d.get('images', []):
        image_file_by_id[img_meta['id']] = Path(img_meta.get('file_name', '')).name

SAMPLES_PER_FORM = 6

for form_name in form_dist['pill_form'].tolist():
    form_class_ids = set(class_df[class_df['pill_form'] == form_name]['class_id'])
    candidates = (
        bbox_df[bbox_df['class_id'].isin(form_class_ids)]
        .drop_duplicates('class_id')
        .head(SAMPLES_PER_FORM)
    )
    if candidates.empty:
        continue

    cols = min(len(candidates), 3)
    rows = int(np.ceil(len(candidates) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 5))
    axes = np.array(axes).flatten()

    for ax, (_, row) in zip(axes, candidates.iterrows()):
        fname    = image_file_by_id.get(row['image_id'])
        img_path = image_path_by_name.get(fname) if fname else None

        if img_path and img_path.exists():
            img = Image.open(img_path).convert('RGB')
            x, y, bw, bh = row['bbox_x'], row['bbox_y'], row['bbox_width'], row['bbox_height']
            pad  = 20
            crop = img.crop((
                max(0, int(x - pad)), max(0, int(y - pad)),
                min(img.width, int(x + bw + pad)), min(img.height, int(y + bh + pad))
            ))
            ax.imshow(crop)
        else:
            ax.text(0.5, 0.5, '이미지 없음', ha='center', va='center', transform=ax.transAxes)

        ax.set_title(f"{row['class_id']}\n{row['class_name']}", fontsize=10)
        ax.axis('off')

    for ax in axes[len(candidates):]:
        ax.axis('off')

    plt.suptitle(f'약제 형태: {form_name}', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

## 6. Bbox 분석

Kaggle 데이터 bbox 분포(01_data_EDA)와 비교해 도메인 갭 여부를 확인합니다.

In [ ]:
# Bbox 유효성 검사
v = bbox_df.copy()
v['is_x_min_valid']    = v['bbox_x'] >= 0
v['is_y_min_valid']    = v['bbox_y'] >= 0
v['is_x_max_valid']    = (v['bbox_x'] + v['bbox_width'])  <= v['image_width']
v['is_y_max_valid']    = (v['bbox_y'] + v['bbox_height']) <= v['image_height']
v['is_center_x_valid'] = v['center_x_ratio'].between(0, 1)
v['is_center_y_valid'] = v['center_y_ratio'].between(0, 1)

validity_cols  = ['is_x_min_valid', 'is_y_min_valid', 'is_x_max_valid',
                  'is_y_max_valid', 'is_center_x_valid', 'is_center_y_valid']
v['is_valid']  = v[validity_cols].all(axis=1)
invalid_df     = v[~v['is_valid']]

validity_summary = pd.DataFrame([
    {'status': 'valid',   'bbox_count': v['is_valid'].sum()},
    {'status': 'invalid', 'bbox_count': len(invalid_df)},
])
validity_summary['ratio'] = validity_summary['bbox_count'] / len(v)
display(validity_summary)

if not invalid_df.empty:
    display(invalid_df[['image_file_name', 'class_id', 'class_name',
                         'image_width', 'image_height',
                         'bbox_x', 'bbox_y', 'bbox_width', 'bbox_height']])

In [ ]:
valid_bbox = bbox_df[v['is_valid']]

summary_cols = ['bbox_width', 'bbox_height', 'bbox_area',
                'aspect_ratio', 'bbox_width_ratio', 'bbox_height_ratio', 'bbox_area_ratio']
display(
    valid_bbox[summary_cols]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])
    .T.reset_index()
    .rename(columns={'index': 'metric', '50%': 'median'})
)

small_thresh = 0.01
large_thresh = 0.10
print(f"Small (<{small_thresh:.0%}): {(valid_bbox['bbox_area_ratio'] < small_thresh).sum():,}")
print(f"Large (>{large_thresh:.0%}): {(valid_bbox['bbox_area_ratio'] > large_thresh).sum():,}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 면적 비율
axes[0].hist(valid_bbox['bbox_area_ratio'], bins=40, color='steelblue', alpha=0.8)
axes[0].axvline(valid_bbox['bbox_area_ratio'].median(), color='red',   linestyle='--',
                label=f'Median: {valid_bbox["bbox_area_ratio"].median():.3f}')
axes[0].axvline(valid_bbox['bbox_area_ratio'].mean(),   color='green', linestyle='-',
                label=f'Mean: {valid_bbox["bbox_area_ratio"].mean():.3f}')
axes[0].set_title('BBox Area Ratio Distribution')
axes[0].set_xlabel('BBox area ratio')
axes[0].set_ylabel('Count')
axes[0].legend()

# 종횡비
axes[1].hist(valid_bbox['aspect_ratio'].dropna(), bins=40, color='darkorange', alpha=0.8)
axes[1].axvline(1.0, color='black', linestyle='--', label='Square (1.0)')
axes[1].axvline(valid_bbox['aspect_ratio'].median(), color='red', linestyle='--',
                label=f'Median: {valid_bbox["aspect_ratio"].median():.2f}')
axes[1].set_title('BBox Aspect Ratio Distribution')
axes[1].set_xlabel('Aspect ratio (w/h)')
axes[1].legend()

# 너비/높이 비율
axes[2].hist(valid_bbox['bbox_width_ratio'].dropna(),  bins=30, alpha=0.6, label='Width ratio')
axes[2].hist(valid_bbox['bbox_height_ratio'].dropna(), bins=30, alpha=0.6, label='Height ratio')
axes[2].set_title('BBox Width/Height Ratio Distribution')
axes[2].set_xlabel('Ratio to image size')
axes[2].legend()

plt.tight_layout()
plt.show()

# 중심 위치 히트맵
plt.figure(figsize=(7, 6))
plt.hist2d(
    valid_bbox['center_x_ratio'].dropna(),
    valid_bbox['center_y_ratio'].dropna(),
    bins=30, range=[[0, 1], [0, 1]], cmap='Blues'
)
plt.colorbar(label='BBox count')
plt.axvline(0.5, color='red', linestyle='--', linewidth=1)
plt.axhline(0.5, color='red', linestyle='--', linewidth=1)
plt.gca().invert_yaxis()
plt.title('BBox Center Position Heatmap')
plt.xlabel('Center x ratio')
plt.ylabel('Center y ratio')
plt.tight_layout()
plt.show()

## 7. Kaggle 대회 데이터셋과 비교

Kaggle 56클래스가 AI Hub 5,000종 중 어디에 해당하는지, overlap이 얼마나 되는지 확인합니다.

In [ ]:
from pathlib import Path

# class_mapping.json 탐색
mapping_path = None
for root in [Path('/content'), Path('/content/drive/MyDrive')]:
    found = list(root.rglob('class_mapping.json'))
    if found:
        mapping_path = found[0]
        break

if mapping_path:
    with mapping_path.open(encoding='utf-8') as f:
        kaggle_mapping = json.load(f)
    kaggle_names = set(kaggle_mapping.keys())
    aihub_names  = set(category_by_id.values())

    overlap     = kaggle_names & aihub_names
    only_kaggle = kaggle_names - aihub_names
    only_aihub  = aihub_names - kaggle_names

    display(pd.DataFrame([
        {'구분': 'Kaggle에만 있음',              '클래스 수': len(only_kaggle)},
        {'구분': 'AI Hub에만 있음',               '클래스 수': len(only_aihub)},
        {'구분': '두 데이터셋 모두 있음 (overlap)', '클래스 수': len(overlap)},
    ]))

    if overlap:
        print('\n겹치는 클래스 예시:', sorted(list(overlap))[:10])
    if only_kaggle:
        print('\nKaggle에만 있는 클래스:', sorted(list(only_kaggle)))
else:
    print('class_mapping.json을 찾을 수 없습니다. Drive 마운트 후 재실행하세요.')

## 8. EDA 요약 및 활용 전략

아래 항목을 직접 채워 넣으세요.

- **총 클래스 / 이미지 수**:
- **약제 형태 비율** (정제 / 경질캡슐 / 연질캡슐):
- **Kaggle 56클래스와 overlap 수 및 비율**:
- **Bbox 크기 분포** — Kaggle 데이터 대비 유사/차이점:
- **Bbox 위치 패턴** — 배치 패턴 촬영 여부:
- **추가 학습 데이터로 활용 시 고려사항** (배경 통제, 조명, 해상도 차이 등):